In [1]:
from enum import Enum, auto

class SequenceStatus(Enum):
    WAITING = auto()
    RUNNING = auto()
    FINISHED = auto()

status = SequenceStatus.WAITING
print(status)

SequenceStatus.WAITING


In [2]:
from dataclasses import dataclass
@dataclass
class Config:
    max_model_len: int = 4096
    tensor_parallel_size: int = 1

cfg = Config(max_model_len=8192)
print(cfg)


Config(max_model_len=8192, tensor_parallel_size=1)


In [1]:
"""
Triton 基本用法示例
Triton 是 OpenAI 开源的 GPU kernel 编程语言，用 Python 写 CUDA 级别的高性能算子。
运行前提：需要 NVIDIA GPU + CUDA，安装 `pip install triton torch`
"""

import torch
import triton
import triton.language as tl


# ============================================================
# 示例 1: 向量加法 (Hello World of Triton)
# ============================================================
@triton.jit
def add_kernel(
    x_ptr,          # 输入张量 x 的指针
    y_ptr,          # 输入张量 y 的指针
    output_ptr,     # 输出张量的指针
    n_elements,     # 元素总数
    BLOCK_SIZE: tl.constexpr,  # 每个 block 处理的元素数（编译期常量）
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask=mask)


def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    n_elements = output.numel()
    # grid: 总共需要多少个 program (block)
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output


# ============================================================
# 示例 2: Softmax (fused kernel，一次读写完成)
# ============================================================
@triton.jit
def softmax_kernel(
    output_ptr, input_ptr,
    input_row_stride, output_row_stride,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    row_idx = tl.program_id(0)
    row_start_ptr = input_ptr + row_idx * input_row_stride
    col_offsets = tl.arange(0, BLOCK_SIZE)
    input_ptrs = row_start_ptr + col_offsets
    mask = col_offsets < n_cols

    row = tl.load(input_ptrs, mask=mask, other=-float("inf"))
    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_output = numerator / denominator

    output_row_start_ptr = output_ptr + row_idx * output_row_stride
    output_ptrs = output_row_start_ptr + col_offsets
    tl.store(output_ptrs, softmax_output, mask=mask)


def softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    y = torch.empty_like(x)
    softmax_kernel[(n_rows,)](
        y, x,
        x.stride(0), y.stride(0),
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return y


# ============================================================
# 示例 3: 矩阵乘法 (展示 2D 分块 + autotune)
# ============================================================
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32}, num_warps=8),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k, other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=mask)


def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]), triton.cdiv(N, meta["BLOCK_N"]))
    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c


# ============================================================
# 验证 & 性能对比
# ============================================================
def main():
    if not torch.cuda.is_available():
        print("需要 CUDA GPU 才能运行 Triton")
        return

    device = "cuda"
    torch.manual_seed(0)

    # 验证向量加法
    size = 98432
    x = torch.rand(size, device=device)
    y = torch.rand(size, device=device)
    out_triton = add(x, y)
    out_torch = x + y
    print(f"[add] max diff: {(out_triton - out_torch).abs().max().item():.2e}")

    # 验证 softmax
    x = torch.randn(1823, 781, device=device)
    out_triton = softmax(x)
    out_torch = torch.softmax(x, dim=1)
    print(f"[softmax] max diff: {(out_triton - out_torch).abs().max().item():.2e}")

    # 验证矩阵乘法
    a = torch.randn(512, 256, device=device, dtype=torch.float32)
    b = torch.randn(256, 384, device=device, dtype=torch.float32)
    out_triton = matmul(a, b)
    out_torch = a @ b
    print(f"[matmul] max diff: {(out_triton - out_torch).abs().max().item():.2e}")


if __name__ == "__main__":
    main()


[add] max diff: 0.00e+00
[softmax] max diff: 7.45e-09
[matmul] max diff: 5.87e-02


In [6]:
@triton.jit
def test():
    print(tl.arange(0, 2))
test[(1,)]()

CompilationError: at 2:4:
def test():
    print(tl.arange(0, 2))
    ^